In [ ]:
import pandas as pd
import numpy as np
import gzip
import os
from tqdm import tqdm

In [2]:
metadata_path = "CRyPTIC_reuse_table_20240917.csv"

df = pd.read_csv(metadata_path)

print("Total samples:", len(df))
df.head()

Total samples: 12287


,ENA_RUN,UNIQUEID,AMI_BINARY_PHENOTYPE,BDQ_BINARY_PHENOTYPE,CFZ_BINARY_PHENOTYPE,DLM_BINARY_PHENOTYPE,EMB_BINARY_PHENOTYPE,ETH_BINARY_PHENOTYPE,INH_BINARY_PHENOTYPE,KAN_BINARY_PHENOTYPE,...,INH_PHENOTYPE_QUALITY,KAN_PHENOTYPE_QUALITY,LEV_PHENOTYPE_QUALITY,LZD_PHENOTYPE_QUALITY,MXF_PHENOTYPE_QUALITY,RIF_PHENOTYPE_QUALITY,RFB_PHENOTYPE_QUALITY,ENA_SAMPLE,VCF,REGENOTYPED_VCF
0,ERR4810489,site.02.subj.0001.lab.2014222001.iso.1,S,NaN,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298516,../reproducibility/00/01/08/61/10861/site.02.i...,../reproducibility/00/01/08/61/10861/site.02.i...
1,ERR4810491,site.02.subj.0002.lab.2014222005.iso.1,S,S,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,LOW,HIGH,HIGH,ERS5298518,../reproducibility/00/01/08/63/10863/site.02.i...,../reproducibility/00/01/08/63/10863/site.02.i...
2,ERR4810493,site.02.subj.0004.lab.2014222010.iso.1,S,S,S,NaN,S,I,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298520,../reproducibility/00/01/08/67/10867/site.02.i...,../reproducibility/00/01/08/67/10867/site.02.i...
3,ERR4810494,site.02.subj.0005.lab.2014222011.iso.1,S,S,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298521,../reproducibility/00/01/08/68/10868/site.02.i...,../reproducibility/00/01/08/68/10868/site.02.i...
4,ERR4810495,site.02.subj.0006.lab.2014222013.iso.1,S,S,S,S,S,S,S,S,...,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,HIGH,ERS5298522,../reproducibility/00/01/08/69/10869/site.02.i...,../reproducibility/00/01/08/69/10869/site.02.i...


In [3]:
def convert_label(x):
    if x in (1, "1"):
        return 1
    if x in (0, "0"):
        return 0

    if pd.isna(x):
        return np.nan

    x = str(x).strip().upper()
    if x == "R":
        return 1
    if x == "S":
        return 0
    return np.nan

In [4]:
drug_columns = [
    "AMI_BINARY_PHENOTYPE",
    "BDQ_BINARY_PHENOTYPE",
    "CFZ_BINARY_PHENOTYPE",
    "DLM_BINARY_PHENOTYPE",
    "EMB_BINARY_PHENOTYPE",
    "ETH_BINARY_PHENOTYPE",
    "INH_BINARY_PHENOTYPE",
    "KAN_BINARY_PHENOTYPE",
    "LEV_BINARY_PHENOTYPE",
    "LZD_BINARY_PHENOTYPE",
    "MXF_BINARY_PHENOTYPE",
    "RFB_BINARY_PHENOTYPE",
    "RIF_BINARY_PHENOTYPE",
]

for col in drug_columns:
    df[col] = df[col].apply(convert_label)

df = df.dropna(subset=drug_columns)

print("Samples after filtering:", len(df))

Samples after filtering: 8749


In [5]:
VCF_DIR = "vcf(2024)"

def extract_variants(vcf_path):
    variants = set()

    with gzip.open(vcf_path, "rt") as f:
        for line in f:
            if line.startswith("#"):
                continue

            parts = line.strip().split("\t")
            if len(parts) < 8:
                continue

            pos = int(parts[1])
            ref = parts[3]
            alt_field = parts[4]
            filt = parts[6]

            if filt not in ("PASS", "."):
                continue

            for alt in alt_field.split(","):
                alt = alt.strip()
                if alt:
                    variants.add((pos, ref, alt))

    return variants

In [6]:
sample_mutations = {}

for idx, row in tqdm(df.iterrows(), total=len(df)):
    sample = row["ENA_SAMPLE"]
    vcf_path = os.path.join(VCF_DIR, f"{sample}.vcf.gz")

    if not os.path.exists(vcf_path):
        continue

    variants = extract_variants(vcf_path)
    sample_mutations[sample] = variants

print("Samples processed:", len(sample_mutations))

100%|██████████| 8749/8749 [00:12<00:00, 678.52it/s]

Samples processed: 8749


In [7]:
GENOME_SIZE = 4411532
WINDOW_SIZE = 100
NUM_WINDOWS = (GENOME_SIZE - 1) // WINDOW_SIZE + 1

print("Window size:", WINDOW_SIZE)
print("Total windows:", NUM_WINDOWS)

Window size: 100
Total windows: 44116


In [8]:
mutation_counts = {}
for muts in sample_mutations.values():
    for variant in muts:
        mutation_counts[variant] = mutation_counts.get(variant, 0) + 1

print("Total variants before filtering:", len(mutation_counts))

MIN_COUNT = 2
filtered_mutations = {v for v, c in mutation_counts.items() if c >= MIN_COUNT}

print(f"Variants kept with count >= {MIN_COUNT}: {len(filtered_mutations)}")

Total variants before filtering: 643599
Variants kept with count >= 2: 165134


In [9]:
matrix = []
samples = []

for sample, variants in tqdm(sample_mutations.items()):
    window_vector = np.zeros(NUM_WINDOWS, dtype=np.float32)

    for pos, ref, alt in variants:
        if (pos, ref, alt) not in filtered_mutations:
            continue
        idx = (pos - 1) // WINDOW_SIZE
        if 0 <= idx < NUM_WINDOWS:
            window_vector[idx] += 1

    window_vector = np.log1p(window_vector)
    matrix.append(window_vector)
    samples.append(sample)

X = pd.DataFrame(matrix, index=samples)
labels = df.set_index("ENA_SAMPLE")[drug_columns].loc[X.index]

print("Feature matrix shape:", X.shape)
print("Labels shape:", labels.shape)

100%|██████████| 8749/8749 [00:05<00:00, 1588.41it/s]


Feature matrix shape: (8749, 44116)
Labels shape: (8749, 13)


In [10]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# 70% train, 15% validation, 15% holdout test
y_all_int = labels.values.astype(np.int64)
msss_1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_idx, temp_idx = next(msss_1.split(X.values, y_all_int))

X_train = X.iloc[train_idx].copy()
y_train = labels.iloc[train_idx].copy()
X_temp = X.iloc[temp_idx].copy()
y_temp = labels.iloc[temp_idx].copy()

msss_2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_rel_idx, test_rel_idx = next(msss_2.split(X_temp.values, y_temp.values.astype(np.int64)))

X_val = X_temp.iloc[val_rel_idx].copy()
y_val = y_temp.iloc[val_rel_idx].copy()
X_test = X_temp.iloc[test_rel_idx].copy()
y_test = y_temp.iloc[test_rel_idx].copy()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (6124, 44116)
Validation: (1312, 44116)
Test: (1313, 44116)


In [11]:
import subprocess
import torch
import torch.nn as nn
from dataclasses import dataclass

def select_device(preferred_gpu=None):
    if not torch.cuda.is_available():
        return torch.device("cpu")

    if preferred_gpu is None:
        env_gpu = os.getenv("WDNN_GPU_INDEX")
        if env_gpu is not None and env_gpu.isdigit():
            preferred_gpu = int(env_gpu)

    gpu_count = torch.cuda.device_count()
    if preferred_gpu is not None and 0 <= preferred_gpu < gpu_count:
        return torch.device(f"cuda:{preferred_gpu}")

    try:
        output = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],
            text=True,
        ).strip().splitlines()
        free_by_gpu = []
        for line in output:
            idx_str, free_str = [x.strip() for x in line.split(",")[:2]]
            free_by_gpu.append((int(idx_str), int(free_str)))
        if free_by_gpu:
            best_gpu = max(free_by_gpu, key=lambda x: x[1])[0]
            return torch.device(f"cuda:{best_gpu}")
    except Exception:
        pass

    return torch.device("cuda:0")

device = select_device()
print("Using device:", device)

@dataclass(frozen=True)
class WDNNConfig:
    input_dim: int = 86392
    hidden_dim: int = 1000
    output_dim: int = 1
    dropout_rate: float = 0.3
    bn_epsilon: float = 1e-3
    bn_momentum_keras: float = 0.99
    kernel_l1: float = 1e-4
    hidden_bias_l2: float = 1e-3
    output_bias_l2: float = 1e-1

class ExactWDNN(nn.Module):
    def __init__(self, config: WDNNConfig) -> None:
        super().__init__()
        self.config = config
        bn_momentum_torch = 1.0 - self.config.bn_momentum_keras

        self.dense1 = nn.Linear(self.config.input_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout = nn.Dropout(p=self.config.dropout_rate)

        self.dense2 = nn.Linear(self.config.hidden_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization_1 = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout_1 = nn.Dropout(p=self.config.dropout_rate)

        self.dense3 = nn.Linear(self.config.hidden_dim, self.config.hidden_dim, bias=True)
        self.batch_normalization_2 = nn.BatchNorm1d(
            self.config.hidden_dim,
            eps=self.config.bn_epsilon,
            momentum=bn_momentum_torch,
            affine=True,
            track_running_stats=True,
        )
        self.dropout_2 = nn.Dropout(p=self.config.dropout_rate)

        self.wdnn_final_layer = nn.Linear(self.config.hidden_dim, self.config.output_dim, bias=True)
        self.output_activation = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.dense1(x)
        x = torch.relu(x)
        x = self.batch_normalization(x)
        x = self.dropout(x)

        x = self.dense2(x)
        x = torch.relu(x)
        x = self.batch_normalization_1(x)
        x = self.dropout_1(x)

        x = self.dense3(x)
        x = torch.relu(x)
        x = self.batch_normalization_2(x)
        x = self.dropout_2(x)

        x = self.wdnn_final_layer(x)
        x = self.output_activation(x)
        return x

    def regularization_loss(self) -> torch.Tensor:
        loss = torch.zeros((), dtype=self.dense1.weight.dtype, device=self.dense1.weight.device)
        for layer in [self.dense1, self.dense2, self.dense3]:
            loss = loss + self.config.kernel_l1 * layer.weight.abs().sum()
            loss = loss + self.config.hidden_bias_l2 * layer.bias.pow(2).sum()

        loss = loss + self.config.kernel_l1 * self.wdnn_final_layer.weight.abs().sum()
        loss = loss + self.config.output_bias_l2 * self.wdnn_final_layer.bias.pow(2).sum()
        return loss

Using device: cuda:1


In [12]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
import copy

N_FOLDS = 10
VAL_SIZE = 0.20
RANDOM_STATE = 42
EPOCHS = 20
BATCH_SIZE = 512
LEARNING_RATE = 1e-3
DEFAULT_THRESHOLD = 0.5

SAVE_DRUG_CHECKPOINTS = True
DRUG_CKPT_DIR = "checkpoints_cv_individual"
if SAVE_DRUG_CHECKPOINTS:
    os.makedirs(DRUG_CKPT_DIR, exist_ok=True)

X_dev = pd.concat([X_train, X_val], axis=0)
y_dev = pd.concat([y_train, y_val], axis=0)
common_idx = X_dev.index.intersection(y_dev.index)
X_dev = X_dev.loc[common_idx]
y_dev = y_dev.loc[common_idx, drug_columns]
feature_columns = list(X_dev.columns)

print("Development split shape:", X_dev.shape)
print("Holdout test split shape:", X_test.shape)

def safe_div(num, den):
    return float(num / den) if den > 0 else np.nan

def tune_binary_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 19)
    best_thr = DEFAULT_THRESHOLD
    best_f1 = -1.0
    best_rec = -1.0

    if len(np.unique(y_true)) < 2:
        return DEFAULT_THRESHOLD

    for thr in thresholds:
        yp = (y_prob >= thr).astype(np.int64)
        tp = int(((yp == 1) & (y_true == 1)).sum())
        fp = int(((yp == 1) & (y_true == 0)).sum())
        fn = int(((yp == 0) & (y_true == 1)).sum())

        prec = safe_div(tp, tp + fp)
        rec = safe_div(tp, tp + fn)
        if np.isnan(prec):
            prec = 0.0
        if np.isnan(rec):
            rec = 0.0
        f1 = safe_div(2 * prec * rec, prec + rec)
        if np.isnan(f1):
            f1 = 0.0

        if (f1 > best_f1) or (np.isclose(f1, best_f1) and rec > best_rec):
            best_f1 = f1
            best_rec = rec
            best_thr = float(thr)

    return best_thr

def compute_binary_metrics(y_true, y_prob, threshold):
    yp = (y_prob >= threshold).astype(np.int64)

    tp = int(((yp == 1) & (y_true == 1)).sum())
    tn = int(((yp == 0) & (y_true == 0)).sum())
    fp = int(((yp == 1) & (y_true == 0)).sum())
    fn = int(((yp == 0) & (y_true == 1)).sum())

    sensitivity = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    precision = safe_div(tp, tp + fp)
    npv = safe_div(tn, tn + fn)

    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan

    return {
        "AUC": auc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "NPV": npv,
        "predicted_positive": int((yp == 1).sum()),
        "actual_positive": int((y_true == 1).sum()),
    }

drug_fold_rows = []
for drug in drug_columns:
    print(f"\n================ {drug} ================")

    y_drug = y_dev[drug].values.astype(np.int64)
    X_np = X_dev.values.astype(np.float32)

    if len(np.unique(y_drug)) < 2:
        print("Skipping drug: only one class in development split")
        continue

    sss = StratifiedShuffleSplit(
        n_splits=N_FOLDS,
        test_size=VAL_SIZE,
        random_state=RANDOM_STATE,
    )

    for fold, (train_idx, val_idx) in enumerate(sss.split(X_np, y_drug), start=1):
        X_train_fold = torch.tensor(X_np[train_idx], dtype=torch.float32)
        y_train_fold = torch.tensor(y_drug[train_idx].reshape(-1, 1), dtype=torch.float32)
        X_val_fold = torch.tensor(X_np[val_idx], dtype=torch.float32)
        y_val_fold = torch.tensor(y_drug[val_idx].reshape(-1, 1), dtype=torch.float32)

        train_loader = DataLoader(TensorDataset(X_train_fold, y_train_fold), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(TensorDataset(X_val_fold, y_val_fold), batch_size=BATCH_SIZE, shuffle=False)

        model = ExactWDNN(WDNNConfig(input_dim=X_np.shape[1], output_dim=1)).to(device)
        criterion = nn.BCELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

        best_val_loss = float("inf")
        best_epoch = 0
        best_state_dict = None
        best_metrics = None
        best_threshold = DEFAULT_THRESHOLD

        for epoch in range(1, EPOCHS + 1):
            model.train()
            train_loss_sum = 0.0
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                preds = model(xb)
                loss = criterion(preds, yb) + model.regularization_loss()
                loss.backward()
                optimizer.step()
                train_loss_sum += loss.item() * xb.size(0)

            train_loss = train_loss_sum / len(train_loader.dataset)

            model.eval()
            val_loss_sum = 0.0
            val_probs_list = []
            val_targets_list = []
            with torch.no_grad():
                for xb, yb in val_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    preds = model(xb)
                    loss = criterion(preds, yb) + model.regularization_loss()
                    val_loss_sum += loss.item() * xb.size(0)
                    val_probs_list.append(preds.detach().cpu().numpy().reshape(-1))
                    val_targets_list.append(yb.detach().cpu().numpy().reshape(-1))

            val_loss = val_loss_sum / len(val_loader.dataset)
            val_probs = np.concatenate(val_probs_list)
            val_targets = np.concatenate(val_targets_list).astype(np.int64)
            epoch_threshold = tune_binary_threshold(val_targets, val_probs)
            epoch_metrics = compute_binary_metrics(val_targets, val_probs, epoch_threshold)

            print(
                f"Drug {drug} Fold {fold} Epoch {epoch}/{EPOCHS} | "
                f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | "
                f"AUC: {epoch_metrics['AUC']:.4f} | Sens: {epoch_metrics['sensitivity']:.4f} | "
                f"Spec: {epoch_metrics['specificity']:.4f} | Prec: {epoch_metrics['precision']:.4f}"
            )

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch
                best_state_dict = copy.deepcopy(model.state_dict())
                best_metrics = epoch_metrics
                best_threshold = epoch_threshold

        ckpt_path = os.path.join(DRUG_CKPT_DIR, f"wdnn_{drug}_fold_{fold:02d}_best.pt")
        if SAVE_DRUG_CHECKPOINTS and best_state_dict is not None:
            torch.save(
                {
                    "drug": drug,
                    "fold": fold,
                    "best_epoch": best_epoch,
                    "model_state_dict": best_state_dict,
                    "config": WDNNConfig(input_dim=X_np.shape[1], output_dim=1),
                    "best_val_loss": best_val_loss,
                    "best_threshold": best_threshold,
                    "best_metrics": best_metrics,
                    "feature_columns": feature_columns,
                },
                ckpt_path,
            )

        row = {
            "drug": drug,
            "fold": fold,
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "threshold": best_threshold,
            **best_metrics,
            "checkpoint": ckpt_path,
        }
        drug_fold_rows.append(row)

individual_fold_df = pd.DataFrame(drug_fold_rows)
print("\n===== Per-drug fold summary =====")
display(individual_fold_df)

Development split shape: (7436, 44116)
Holdout test split shape: (1313, 44116)

================ AMI_BINARY_PHENOTYPE ================
Drug AMI_BINARY_PHENOTYPE Fold 1 Epoch 1/20 | Train loss: 10.1536 | Val loss: 7.9671 | AUC: 0.4834 | Sens: 0.2208 | Spec: 0.8554 | Prec: 0.0769
Drug AMI_BINARY_PHENOTYPE Fold 1 Epoch 2/20 | Train loss: 6.9531 | Val loss: 5.8358 | AUC: 0.7465 | Sens: 0.1688 | Spec: 0.9965 | Prec: 0.7222
Drug AMI_BINARY_PHENOTYPE Fold 1 Epoch 3/20 | Train loss: 5.0343 | Val loss: 4.4016 | AUC: 0.7526 | Sens: 0.1688 | Spec: 0.9965 | Prec: 0.7222
Drug AMI_BINARY_PHENOTYPE Fold 1 Epoch 4/20 | Train loss: 3.8832 | Val loss: 3.4429 | AUC: 0.7787 | Sens: 0.2078 | Spec: 0.9965 | Prec: 0.7619
Drug AMI_BINARY_PHENOTYPE Fold 1 Epoch 5/20 | Train loss: 3.1590 | Val loss: 2.8838 | AUC: 0.8411 | Sens: 0.2078 | Spec: 0.9965 | Prec: 0.7619
Drug AMI_BINARY_PHENOTYPE Fold 1 Epoch 6/20 | Train loss: 2.7429 | Val loss: 2.5350 | AUC: 0.8024 | Sens: 0.3506 | Spec: 0.9872 | Prec: 0.6000
Drug A

,drug,fold,best_epoch,best_val_loss,threshold,AUC,sensitivity,specificity,precision,NPV,predicted_positive,actual_positive,checkpoint
0,AMI_BINARY_PHENOTYPE,1,20,1.465489,0.10,0.874962,0.454545,0.995039,0.833333,0.970954,42,77,checkpoints_cv_individual/wdnn_AMI_BINARY_PHEN...
1,AMI_BINARY_PHENOTYPE,2,19,1.507024,0.05,0.784127,0.207792,0.995039,0.695652,0.958362,23,77,checkpoints_cv_individual/wdnn_AMI_BINARY_PHEN...
2,AMI_BINARY_PHENOTYPE,3,19,1.508565,0.05,0.628752,0.025974,1.000000,1.000000,0.949529,2,77,checkpoints_cv_individual/wdnn_AMI_BINARY_PHEN...
3,AMI_BINARY_PHENOTYPE,4,19,1.507083,0.05,0.874428,0.077922,0.997874,0.666667,0.951995,9,77,checkpoints_cv_individual/wdnn_AMI_BINARY_PHEN...
4,AMI_BINARY_PHENOTYPE,5,17,1.501643,0.05,0.813681,0.545455,0.909993,0.248521,0.973465,169,77,checkpoints_cv_individual/wdnn_AMI_BINARY_PHEN...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,RIF_BINARY_PHENOTYPE,6,20,1.703561,0.10,0.953623,0.906250,0.889423,0.779271,0.956567,521,448,checkpoints_cv_individual/wdnn_RIF_BINARY_PHEN...
126,RIF_BINARY_PHENOTYPE,7,19,1.760129,0.10,0.922105,0.787946,0.929808,0.828638,0.910546,426,448,checkpoints_cv_individual/wdnn_RIF_BINARY_PHEN...
127,RIF_BINARY_PHENOTYPE,8,18,1.715836,0.15,0.959907,0.837054,0.960577,0.901442,0.931903,416,448,checkpoints_cv_individual/wdnn_RIF_BINARY_PHEN...
128,RIF_BINARY_PHENOTYPE,9,19,1.591908,0.20,0.954625,0.859375,0.949038,0.878995,0.940000,438,448,checkpoints_cv_individual/wdnn_RIF_BINARY_PHEN...


In [13]:
# Evaluate best checkpoint per drug on holdout test split

def load_checkpoint(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

if "individual_fold_df" not in globals() or individual_fold_df.empty:
    raise RuntimeError("individual_fold_df not found. Run the training cell first.")

common_idx = X_test.index.intersection(y_test.index)
X_test_eval = X_test.loc[common_idx]
y_test_eval = y_test.loc[common_idx, drug_columns]

test_rows = []
for drug in drug_columns:
    drug_df = individual_fold_df[individual_fold_df["drug"] == drug]
    if drug_df.empty:
        continue

    best_row = drug_df.sort_values(by=["AUC", "fold"], ascending=[False, True]).iloc[0]
    ckpt_path = best_row["checkpoint"]
    ckpt = load_checkpoint(ckpt_path, map_location="cpu")

    feature_cols = ckpt.get("feature_columns")
    if feature_cols is None:
        X_drug = X_test_eval
    else:
        missing_cols = [c for c in feature_cols if c not in X_test_eval.columns]
        if missing_cols:
            print(f"Skipping {drug}: missing feature columns in test set")
            continue
        X_drug = X_test_eval.loc[:, feature_cols]

    cfg = ckpt.get("config")
    if isinstance(cfg, dict):
        model_cfg = WDNNConfig(
            input_dim=int(cfg.get("input_dim", X_drug.shape[1])),
            hidden_dim=int(cfg.get("hidden_dim", 1000)),
            output_dim=1,
            dropout_rate=float(cfg.get("dropout_rate", 0.3)),
            bn_epsilon=float(cfg.get("bn_epsilon", 1e-3)),
            bn_momentum_keras=float(cfg.get("bn_momentum_keras", 0.99)),
            kernel_l1=float(cfg.get("kernel_l1", 1e-4)),
            hidden_bias_l2=float(cfg.get("hidden_bias_l2", 1e-3)),
            output_bias_l2=float(cfg.get("output_bias_l2", 1e-1)),
        )
    else:
        model_cfg = WDNNConfig(input_dim=X_drug.shape[1], output_dim=1)

    model = ExactWDNN(model_cfg)
    eval_device = device
    try:
        model = model.to(eval_device)
    except torch.OutOfMemoryError:
        eval_device = torch.device("cpu")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        model = model.to(eval_device)

    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    X_tensor = torch.tensor(X_drug.values.astype(np.float32), dtype=torch.float32).to(eval_device)
    with torch.no_grad():
        probs = model(X_tensor).cpu().numpy().reshape(-1)

    y_true = y_test_eval[drug].values.astype(np.int64)
    threshold = float(ckpt.get("best_threshold", DEFAULT_THRESHOLD))
    metrics = compute_binary_metrics(y_true, probs, threshold)

    test_rows.append(
        {
            "drug": drug,
            "best_fold": int(best_row["fold"]),
            "threshold": threshold,
            **metrics,
        }
    )

individual_test_df = pd.DataFrame(test_rows).sort_values("drug").reset_index(drop=True)
print("\n===== Holdout test metrics (best checkpoint per drug) =====")
display(individual_test_df)

if not individual_test_df.empty:
    macro = individual_test_df[["AUC", "sensitivity", "specificity", "precision", "NPV"]].mean(numeric_only=True)
    print("\n===== Macro average across drugs =====")
    display(pd.DataFrame({"macro_mean": macro}))


===== Holdout test metrics (best checkpoint per drug) =====


,drug,best_fold,threshold,AUC,sensitivity,specificity,precision,NPV,predicted_positive,actual_positive
0,AMI_BINARY_PHENOTYPE,1,0.10,0.841873,0.367647,0.998394,0.925926,0.966563,27,68
1,BDQ_BINARY_PHENOTYPE,9,0.05,0.697698,0.000000,1.000000,NaN,0.992384,0,10
2,CFZ_BINARY_PHENOTYPE,4,0.05,0.663998,0.000000,1.000000,NaN,0.958111,0,55
3,DLM_BINARY_PHENOTYPE,10,0.05,0.505620,0.000000,1.000000,NaN,0.986291,0,18
4,EMB_BINARY_PHENOTYPE,7,0.20,0.956582,0.784906,0.969466,0.866667,0.946878,240,265
5,ETH_BINARY_PHENOTYPE,2,0.15,0.917313,0.558011,0.981449,0.827869,0.932830,122,181
6,INH_BINARY_PHENOTYPE,10,0.50,0.937836,0.875240,0.911616,0.866920,0.917408,526,521
7,KAN_BINARY_PHENOTYPE,2,0.10,0.846038,0.393258,0.991013,0.760870,0.957380,46,89
8,LEV_BINARY_PHENOTYPE,3,0.10,0.880835,0.629213,0.930396,0.586387,0.941176,191,178
9,LZD_BINARY_PHENOTYPE,1,0.05,0.620642,0.000000,1.000000,NaN,0.989337,0,14



===== Macro average across drugs =====


,macro_mean
AUC,0.819876
sensitivity,0.459426
specificity,0.971073
precision,0.796249
NPV,0.956158
